# 01 Weekend Data and Feature Engineering (AMZN/MSFT)

This notebook constructs the **decision panel** for weekend-return forecasting.

- Universe: AMZN, MSFT
- Frequency: daily raw OHLCV
- Decision timestamp: Friday close
- Primary target: Friday close to next trading day close
- Robustness target: Friday close to next trading day open


## Academic Context (Seminal + practical)

- French (1980): weekend effects in stock returns.
- Friedman (2001): gradient boosting foundations.
- Chen & Guestrin (2016): XGBoost scalable boosting system.
- Gu, Kelly, Xiu (2020): ML in asset pricing.
- Moody & Saffell (2001), Jiang et al. (2017): RL in trading/portfolio context.


In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
OUT_DIR = PROJECT_ROOT / "research_outputs" / "weekend_rl_xgb"
DATA_CACHE_DIR = OUT_DIR / "data_cache"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
for p in [OUT_DIR, DATA_CACHE_DIR, TABLE_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "tickers": ["AMZN", "MSFT"],
    "start": "2010-01-01",
    "end": None,
    "warmup_days": 80,
}
CONFIG


In [ ]:
def download_daily_ohlcv(ticker: str, start: str, end: str | None = None) -> pd.DataFrame:
    df = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        actions=True,
    )
    if df.empty:
        raise ValueError(f"No data downloaded for {ticker}")
    df = df.reset_index().rename(columns=str)
    if "Adj Close" not in df.columns:
        df["Adj Close"] = df["Close"]
    df["ticker"] = ticker
    df = df.sort_values("Date").reset_index(drop=True)
    return df

all_raw = []
for t in CONFIG["tickers"]:
    raw_t = download_daily_ohlcv(t, CONFIG["start"], CONFIG["end"])
    all_raw.append(raw_t)

raw_df = pd.concat(all_raw, ignore_index=True)
raw_df.to_parquet(DATA_CACHE_DIR / "raw_daily_ohlcv.parquet", index=False)
raw_df.head()


In [ ]:
def rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(alpha=1 / window, adjust=False).mean()
    roll_down = down.ewm(alpha=1 / window, adjust=False).mean()
    rs = roll_up / (roll_down + 1e-12)
    return 100 - (100 / (1 + rs))


def add_features_symbol(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values("Date")
    px = df["Adj Close"].astype(float)
    close = df["Close"].astype(float)
    high = df["High"].astype(float)
    low = df["Low"].astype(float)
    vol = df["Volume"].astype(float)

    df["ret_1d"] = px.pct_change()
    for w in [5, 10, 21, 63]:
        df[f"ret_{w}d"] = px.pct_change(w)
        df[f"sma_{w}"] = px.rolling(w).mean()
        df[f"ema_{w}"] = px.ewm(span=w, adjust=False).mean()
        df[f"vol_{w}"] = df["ret_1d"].rolling(w).std()

    ema12 = px.ewm(span=12, adjust=False).mean()
    ema26 = px.ewm(span=26, adjust=False).mean()
    df["macd"] = ema12 - ema26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_hist"] = df["macd"] - df["macd_signal"]

    df["rsi_14"] = rsi(px, 14)

    tr_components = pd.concat([
        (high - low).rename("hl"),
        (high - close.shift(1)).abs().rename("hc"),
        (low - close.shift(1)).abs().rename("lc"),
    ], axis=1)
    true_range = tr_components.max(axis=1)
    df["atr_14"] = true_range.rolling(14).mean()

    bb_mid = px.rolling(20).mean()
    bb_std = px.rolling(20).std()
    df["bb_z_20"] = (px - bb_mid) / (bb_std + 1e-12)
    df["bb_width_20"] = (2 * bb_std) / (bb_mid + 1e-12)

    df["vol_sma_5"] = vol.rolling(5).mean()
    df["vol_sma_21"] = vol.rolling(21).mean()
    df["vol_z_21"] = (vol - df["vol_sma_21"]) / (vol.rolling(21).std() + 1e-12)
    df["dollar_volume"] = close * vol
    df["dollar_vol_sma_21"] = df["dollar_volume"].rolling(21).mean()

    df["dow"] = df["Date"].dt.dayofweek
    df["month"] = df["Date"].dt.month
    df["is_month_end"] = df["Date"].dt.is_month_end.astype(int)
    df["is_quarter_end"] = df["Date"].dt.is_quarter_end.astype(int)
    df["is_friday"] = (df["dow"] == 4).astype(int)

    # Targets computed on Friday decision rows.
    next_idx = np.arange(len(df)) + 1
    valid_next = next_idx < len(df)
    df["next_close"] = np.where(valid_next, px.shift(-1), np.nan)
    df["next_open"] = np.where(valid_next, df["Open"].shift(-1), np.nan)

    df["ret_weekend_close"] = np.where(
        df["is_friday"].eq(1),
        df["next_close"] / px - 1,
        np.nan,
    )
    df["ret_weekend_gap"] = np.where(
        df["is_friday"].eq(1),
        df["next_open"] / close - 1,
        np.nan,
    )

    return df

feat_df = (
    raw_df.groupby("ticker", group_keys=False)
    .apply(add_features_symbol)
    .reset_index(drop=True)
)
feat_df.head()


In [ ]:
# Cross-asset spillover features at daily level, then sampled on Friday rows.
pivot_ret = feat_df.pivot(index="Date", columns="ticker", values="ret_1d").sort_index()
pivot_vol = feat_df.pivot(index="Date", columns="ticker", values="vol_21").sort_index()

cross_map = []
for t in CONFIG["tickers"]:
    other = [x for x in CONFIG["tickers"] if x != t][0]
    tmp = pd.DataFrame({
        "Date": pivot_ret.index,
        "ticker": t,
        f"other_ret_1d_{other.lower()}": pivot_ret[other].values,
        f"other_vol_21_{other.lower()}": pivot_vol[other].values,
    })
    cross_map.append(tmp)
cross_df = pd.concat(cross_map, ignore_index=True)
feat_df = feat_df.merge(cross_df, on=["Date", "ticker"], how="left")


In [ ]:
# Build Friday-only decision panel.
decision_panel = feat_df.loc[feat_df["is_friday"].eq(1)].copy()

# Keep leakage-safe predictors: all are derived from t or earlier; targets use t+1 only.
non_feature_cols = {
    "Date", "ticker", "ret_weekend_close", "ret_weekend_gap", "next_close", "next_open", "Dividends", "Stock Splits"
}
feature_cols = [c for c in decision_panel.columns if c not in non_feature_cols]

# Warmup and NA handling for engineered features.
min_date_by_ticker = decision_panel.groupby("ticker")["Date"].min().to_dict()
keep_mask = pd.Series(True, index=decision_panel.index)
for t, dmin in min_date_by_ticker.items():
    keep_mask &= ~((decision_panel["ticker"] == t) & (decision_panel["Date"] < dmin + pd.Timedelta(days=CONFIG["warmup_days"])))

decision_panel = decision_panel.loc[keep_mask].copy()

na_before = decision_panel[feature_cols + ["ret_weekend_close"]].isna().sum().sum()
decision_panel = decision_panel.dropna(subset=feature_cols + ["ret_weekend_close"]).copy()
na_after = decision_panel[feature_cols + ["ret_weekend_close"]].isna().sum().sum()

print({"na_before": int(na_before), "na_after": int(na_after), "rows": len(decision_panel)})


In [ ]:
# Data integrity and leakage checks.
assert set(decision_panel["ticker"].unique()) == set(CONFIG["tickers"]), "Ticker universe mismatch"
assert decision_panel["Date"].min() >= pd.Timestamp(CONFIG["start"]), "Start date violation"

# Ensure decision rows are Fridays.
assert (decision_panel["Date"].dt.dayofweek == 4).all(), "Non-Friday decision row detected"

# Ensure target exists only because of next trading day mapping.
assert decision_panel["ret_weekend_close"].notna().all(), "Missing primary target values"

# Simple holiday robustness indicator: number of Friday->next calendar day gaps > 3 days.
holiday_gap_df = feat_df.loc[feat_df["is_friday"].eq(1), ["Date", "ticker"]].copy()
holiday_gap_df["next_date"] = holiday_gap_df.groupby("ticker")["Date"].shift(-1)
holiday_gap_df["gap_days"] = (holiday_gap_df["next_date"] - holiday_gap_df["Date"]).dt.days
print("Weekend gaps > 3 calendar days:", int((holiday_gap_df["gap_days"] > 3).sum()))


In [ ]:
# Persist contracts for downstream notebooks.
decision_panel = decision_panel.rename(columns={"Date": "date_decision"}).sort_values(["date_decision", "ticker"]).reset_index(drop=True)

payload = {
    "config": CONFIG,
    "feature_cols": feature_cols,
    "n_rows": int(len(decision_panel)),
    "date_min": str(decision_panel["date_decision"].min().date()),
    "date_max": str(decision_panel["date_decision"].max().date()),
}

(decision_panel).to_parquet(OUT_DIR / "decision_panel.parquet", index=False)
with open(OUT_DIR / "decision_panel_metadata.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print(payload)
